# Track 1 MHR export diagnostic
CPU numerical check using synthetic parameters and the actual MHR model. No challenge reconstruction or scoring output.

In [ ]:
import os,sys,json,subprocess,urllib.request,hashlib,tarfile,shutil
from pathlib import Path
def file_sha256(path):
 digest=hashlib.sha256()
 with path.open('rb') as handle:
  for chunk in iter(lambda:handle.read(16*1024*1024),b''):digest.update(chunk)
 return digest.hexdigest()
manifests=list(Path('/kaggle/input').rglob('runtime_cache_manifest.json'))
assert len(manifests)==1,manifests
manifest=json.loads(manifests[0].read_text())
cache=manifests[0].with_name('track1_runtime.bin')
if not cache.exists():cache=manifests[0].with_name('track1_runtime.tar.gz')
assert cache.stat().st_size==manifest['bytes']
assert file_sha256(cache)==manifest['sha256']
assert manifest['runtime_revision']=='8c778f905cb35f76f8e6cd6e2ea486da4fb9e0ef'
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','gcc-12','g++-12','cmake','libeigen3-dev','libboost-system-dev','libboost-program-options-dev','libegl1-mesa-dev','libosmesa6-dev','libspatialindex-dev','libgl1','ffmpeg'],check=True)
# Only accept the explicit paths listed in the manifest, preventing arbitrary writes.
allowed=tuple(x.lstrip('/').rstrip('/')+'/' for x in manifest['paths'])
with tarfile.open(cache) as archive:
 for item in archive.getmembers():
  name=item.name.rstrip('/')+'/'
  if not name.startswith(allowed):raise ValueError('Unexpected runtime archive member '+item.name)
 archive.extractall('/',filter='fully_trusted')
py='/tmp/track1-env/bin/python'
# uv's venv link uses an unversioned alias that was not part of the cache.
venv_python=Path(py)
interpreter_target=Path(os.readlink(venv_python))
base=Path(manifest['paths'][-1])
assert (base/'bin'/interpreter_target.name).is_file(),'Cached base interpreter missing'
alias=interpreter_target.parent.parent
if not alias.exists():alias.symlink_to(base,target_is_directory=True)
assert venv_python.is_file(),'Python interpreter alias was not restored'
subprocess.run([py,'-c','import sys; assert sys.version_info[:2]==(3,11); print(sys.version)'],check=True)
cuda=Path('/tmp/cuda124')
modules=Path('/tmp/video_to_data/reconstruction/modules')
workspace=Path('/workspace');workspace.mkdir(exist_ok=True)
for name in ['v2d_common','v2d_mv','v2d_mesh','v2d_foundation_pose','v2d_moge','v2d_sam3d_body','v2d_cari4d']:
 path=workspace/name
 if not path.exists():path.symlink_to(modules/name,target_is_directory=True)
env=os.environ.copy();env.update(CUDA_HOME=str(cuda),PATH=str(cuda/'bin')+':'+env['PATH'],CC='gcc-12',CXX='g++-12',TORCH_CUDA_ARCH_LIST='7.5',FORCE_CUDA='1',MAX_JOBS='2',PYOPENGL_PLATFORM='egl',OPENCV_IO_ENABLE_OPENEXR='1',CUB_HOME=str(cuda/'include'),CPATH=str(cuda/'include')+':/usr/include/eigen3')
env['PYTHONPATH']=':'.join(str(workspace/x) for x in ['v2d_cari4d/lib/cari4d','v2d_sam3d_body/lib','v2d_foundation_pose/lib/FoundationPose'])
print('Verified native runtime cache restored',flush=True)


In [ ]:
weights=Path('/tmp/track1_weights');weights.mkdir(exist_ok=True)
manifests=[json.loads(p.read_text()) for p in Path('/kaggle/input').rglob('model_manifest.json')]
manifest=next(value for value in manifests if isinstance(value,list) and len(value)==8 and all('file' in x and 'sha256' in x for x in value))
archives=list(Path('/kaggle/input').rglob('track1_models.tar'))
if len(archives)==1:
 with tarfile.open(archives[0]) as archive:archive.extractall(weights,filter='data')
else:
 # Kaggle expands uploaded .tar files into a directory named after the archive.
 roots=[p.parents[2] for p in Path('/kaggle/input').rglob('step200000.pth') if p.parent.name=='2026-08-25-09-35-57']
 assert len(roots)==1,roots
 for item in manifest:
  source=roots[0]/item['file'];destination=weights/item['file']
  destination.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,destination)
for item in manifest:
 path=weights/item['file'];assert path.stat().st_size==item['bytes']
 assert file_sha256(path)==item['sha256'],path
# T4 lacks bfloat16. Use full precision for the human initializer to avoid FP16 overflow.
config=weights/'sam3d_body/checkpoints/sam-3d-body-dinov3/model_config.yaml'
original_config=config.read_text()
assert 'USE_FP16: true' in original_config
config.write_text(original_config.replace('USE_FP16: true','USE_FP16: false'))
Path('/kaggle/working/t4_precision_manifest.json').write_text(json.dumps({
 'sam3d_precision':'float32','original_config_sha256':hashlib.sha256(original_config.encode()).hexdigest(),
 'effective_config_sha256':hashlib.sha256(config.read_bytes()).hexdigest()},indent=2))
env['HF_HOME']=str(weights/'hf_home')
public_models="""
from pathlib import Path
from v2d.cari4d.lib.download_weights import _download_moge2,_download_torch_hub_models
from v2d.foundation_pose.lib.download_weights import download_weights
from v2d.foundation_pose.lib.backends import NVLABS_PYTORCH
weights=Path('/tmp/track1_weights')
_download_moge2(weights)
_download_torch_hub_models(weights)
download_weights(str(weights/'foundationpose'),backend=NVLABS_PYTORCH)
"""
public_caches=list(Path('/kaggle/input').rglob('public_models.bin'))
if public_caches:
 assert len(public_caches)==1,public_caches
 archive_path=public_caches[0]
 cache_manifest=json.loads(archive_path.with_name('public_model_cache_manifest.json').read_text())
 assert archive_path.stat().st_size==cache_manifest['bytes']
 assert file_sha256(archive_path)==cache_manifest['sha256']
 prefixes=tuple(x.rstrip('/')+'/' for x in cache_manifest['paths'])
 with tarfile.open(archive_path) as archive:
  for item in archive.getmembers():
   if not (item.name.rstrip('/')+'/').startswith(prefixes):raise ValueError('Unexpected public-model cache member')
  archive.extractall(weights,filter='data')
 print('Verified public model cache restored',flush=True)
else:
 subprocess.run([py,'-c',public_models],env=env,check=True)


In [ ]:
code=Path('/tmp/track1_diagnostic_code');code.mkdir(exist_ok=True)
(code/'export_cari4d.py').write_text('"""Export an actual native-MHR CARI4D wild-video result to the official kit.\n\nRun in the reconstruction environment with its PYTHONPATH and MHR assets.\nThis is an adapter, not a reconstruction method. It refuses incomplete timelines\nand validates the exported native parameters by decoding the reference model.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport numpy as np\nimport torch\nfrom lib_mhr import MHRLayer\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        while chunk := f.read(8*1024*1024):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--bundle\', type=Path, required=True)\n    p.add_argument(\'--mesh\', type=Path, required=True)\n    p.add_argument(\'--mhr-assets-root\', type=Path, required=True)\n    p.add_argument(\'--episode\', type=int, required=True)\n    p.add_argument(\'--frames\', type=int, required=True)\n    p.add_argument(\'--output\', type=Path, required=True)\n    p.add_argument(\'--batch-size\', type=int, default=8)\n    p.add_argument(\'--device\', default=\'cuda\')\n    a = p.parse_args()\n    if a.episode < 0 or a.frames < 1 or a.batch_size < 1:\n        p.error(\'Episode, frame count or batch size is invalid\')\n    if a.mesh.name != \'output_aligned.glb\':\n        p.error(\'Use the actual wild-pipeline aligned mesh: output_aligned.glb\')\n    # This bundle is produced by our pinned inference code, not an untrusted upload.\n    bundle = torch.load(a.bundle, map_location=\'cpu\', weights_only=False)\n    if bundle.get(\'metadata\', {}).get(\'ground_truth_used\') is not False:\n        raise ValueError(\'Expected explicit no-ground-truth inference provenance\')\n    frames = [int(f) for f in bundle[\'frames\']]\n    if frames != list(range(a.frames)):\n        raise ValueError(\'Bundle must cover every original video frame in order\')\n    pr = bundle[\'pr\']\n    for key in (\'mhr_global_rot6d\', \'mhr_trans\', \'mhr_body_pose_cont\',\n                \'mhr_hand\', \'mhr_shape\', \'mhr_scale\'):\n        values = np.asarray(pr[key])\n        if len(values) != a.frames or not np.isfinite(values).all():\n            raise ValueError(f\'Invalid or incomplete parameter array: {key}\')\n    poses = np.asarray(pr[\'pose_abs\'], dtype=np.float32)\n    if poses.shape != (a.frames,4,4) or not np.isfinite(poses).all():\n        raise ValueError(\'Expected finite object poses [T,4,4]\')\n    np.testing.assert_allclose(poses[:,3,:], np.tile([0,0,0,1],(a.frames,1)), atol=1e-6)\n    rotations = poses[:,:3,:3]\n    np.testing.assert_allclose(rotations.transpose(0,2,1)@rotations,\n                               np.tile(np.eye(3),(a.frames,1,1)), atol=1e-4)\n    if (np.linalg.det(rotations) <= 0).any():\n        raise ValueError(\'Object pose contains a reflection\')\n    metadata = bundle.get(\'metadata\', {})\n    transform = np.asarray(metadata.get(\'object_mesh_to_training_transform\', np.eye(4)))\n    if not np.allclose(transform, np.eye(4), atol=1e-6):\n        raise ValueError(\'Nonidentity object mesh frame requires explicit geometric conversion\')\n\n    layer = MHRLayer.from_mhr_assets(mhr_assets_root=a.mhr_assets_root, device=a.device)\n    backend = layer.backend\n    model_path = backend.mhr_model_path\n    reference = torch.jit.load(str(model_path), map_location=a.device).eval()\n    flip = torch.tensor([1.,-1.,-1.], device=a.device)\n    # The competition requires one identity for the entire clip.\n    shape = np.median(np.asarray(pr[\'mhr_shape\']), axis=0).astype(np.float32)\n    latent_scale = np.median(np.asarray(pr[\'mhr_scale\']), axis=0).astype(np.float32)\n    exported = []\n    scales = None\n    errors = []\n    for start in range(0,a.frames,a.batch_size):\n        stop = min(start+a.batch_size,a.frames)\n        params = {k: torch.as_tensor(np.asarray(v)[start:stop],device=a.device,dtype=torch.float32)\n                  for k,v in pr.items() if k.startswith(\'mhr_\')}\n        count = stop-start\n        params[\'mhr_shape\'] = torch.as_tensor(shape,device=a.device).expand(count,-1)\n        params[\'mhr_scale\'] = torch.as_tensor(latent_scale,device=a.device).expand(count,-1)\n        # Expression is absent from the challenge schema and the scorer uses zero.\n        params[\'mhr_face\'] = torch.zeros(count,72,device=a.device)\n        with torch.inference_mode():\n            context = backend.prepare_vertices_context(params)\n            translation,body,identity,scale = backend._mutable_vertices_inputs(params,context)\n            zero = torch.zeros_like(translation)\n            verts, native = context.head.mhr_forward(\n                global_trans=zero,global_rot=context.global_rot,body_pose_params=body,\n                hand_pose_params=context.hand,scale_params=scale,shape_params=identity,\n                expr_params=params[\'mhr_face\'],return_model_params=True)\n            target = verts*flip + translation[:,None,:]\n            native = native.clone()\n            expr = torch.zeros(count,72,device=a.device)\n\n            def decode(model_params):\n                v,_ = reference(identity.contiguous(),model_params.contiguous(),expr,True)\n                return v*flip/100.\n\n            base = decode(native)\n            # Determine the native root translation units from the model itself.\n            columns = []\n            for axis in range(3):\n                shifted = native.clone();shifted[:,axis] += 1.\n                columns.append((decode(shifted)-base).mean(dim=1))\n            jacobian = torch.stack(columns,dim=-1)\n            delta = torch.linalg.solve(jacobian,(target-base).mean(dim=1).unsqueeze(-1)).squeeze(-1)\n            native[:,:3] += delta\n            actual = decode(native)\n            error = torch.linalg.vector_norm(actual-target,dim=-1).amax(dim=1)\n            if not torch.isfinite(error).all() or error.max().item() > 0.0005:\n                raise ValueError(f\'Native decoder round trip exceeds 0.5 mm: {error.max().item()} m\')\n        block = native.detach().cpu().numpy()\n        if block.shape != (count,204):\n            raise ValueError(f\'Expected native MHR parameters [B,204], got {block.shape}\')\n        if scales is None:\n            scales = block[0,136:].copy()\n        np.testing.assert_allclose(block[:,136:],np.tile(scales,(count,1)),atol=1e-6)\n        exported.append(block[:,:136]);errors.extend(error.cpu().tolist())\n    a.output.mkdir(parents=True,exist_ok=True)\n    stem = a.output/f\'episode_{a.episode:06d}\'\n    np.savez_compressed(stem.with_suffix(\'.npz\'),pose=np.concatenate(exported),\n        scales=scales,shape=shape,object_rotation=rotations,\n        object_translation=poses[:,:3,3],object_scale=np.float32(1.))\n    mesh_out = a.output/(stem.name+\'_object.glb\')\n    shutil.copy2(a.mesh,mesh_out)\n    report = {\'episode\':a.episode,\'frames\':a.frames,\'source_bundle_sha256\':sha256(a.bundle),\n        \'mesh_sha256\':sha256(mesh_out),\'mhr_model_sha256\':sha256(model_path),\n        \'identity_selection\':\'per-clip median of predicted identity latents\',\n        \'expression\':\'zero, matching challenge scorer\',\n        \'max_native_round_trip_error_m\':max(errors),\'ground_truth_used\':False}\n    stem.with_suffix(\'.report.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(json.dumps(report,indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n')
(code/'check_native_export.py').write_text('"""Numerical MHR exporter diagnostic using synthetic parameters, never submitted.\n\nDecode the actual licensed MHR model and check known translations against the\ncompetition\'s independent TorchScript coordinate convention. Temporary fixture\narrays are removed; only an explicitly labelled diagnostic report is retained.\n"""\nimport argparse,json,shutil,subprocess,sys,tempfile\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom lib_mhr.rotations import rotmat_to_6d\nfrom sam_3d_body.models.modules.mhr_utils import compact_model_params_to_cont_body\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--assets\',type=Path,required=True)\n    p.add_argument(\'--actual-mesh\',type=Path,required=True)\n    p.add_argument(\'--report\',type=Path,required=True)\n    a=p.parse_args()\n    with tempfile.TemporaryDirectory(prefix=\'mhr-export-diagnostic-\') as directory:\n        temporary=Path(directory);frames=3\n        translation=np.array([[0.,0.,2.],[.1,-.05,2.2],[-.2,.15,1.8]],dtype=np.float32)\n        params={\'mhr_global_rot6d\':rotmat_to_6d(torch.eye(3).expand(frames,-1,-1)).numpy(),\n            \'mhr_trans\':translation,\'mhr_body_pose_cont\':compact_model_params_to_cont_body(torch.zeros(frames,133)).numpy(),\n            \'mhr_hand\':np.zeros((frames,108),np.float32),\'mhr_shape\':np.zeros((frames,45),np.float32),\n            \'mhr_scale\':np.zeros((frames,28),np.float32),\'pose_abs\':np.tile(np.eye(4,dtype=np.float32),(frames,1,1))}\n        bundle={\'frames\':[str(i) for i in range(frames)],\'pr\':params,\n                \'metadata\':{\'ground_truth_used\':False,\'synthetic_parameter_diagnostic\':True}}\n        torch.save(bundle,temporary/\'fixture.pth\')\n        mesh=temporary/\'output_aligned.glb\';shutil.copy2(a.actual_mesh,mesh)\n        subprocess.run([sys.executable,str(Path(__file__).with_name(\'export_cari4d.py\')),\n            \'--bundle\',str(temporary/\'fixture.pth\'),\'--mesh\',str(mesh),\n            \'--mhr-assets-root\',str(a.assets),\'--episode\',\'0\',\'--frames\',str(frames),\n            \'--output\',str(temporary/\'diagnostic_only\'),\'--device\',\'cpu\',\'--batch-size\',\'3\'],check=True)\n        output=temporary/\'diagnostic_only/episode_000000.npz\'\n        model=torch.jit.load(str(a.assets/\'checkpoints/sam-3d-body-dinov3/assets/mhr_model.pt\'),map_location=\'cpu\').eval()\n        with np.load(output) as values,torch.inference_mode():\n            native=torch.from_numpy(np.concatenate([values[\'pose\'],np.tile(values[\'scales\'],(frames,1))],axis=1))\n            shape=torch.from_numpy(values[\'shape\']).expand(frames,-1).contiguous()\n            vertices,_=model(shape,native,torch.zeros(frames,72),True)\n            vertices=vertices.numpy()*np.array([1.,-1.,-1.],np.float32)/100.\n        expected=translation[1:]-translation[0]\n        measured=vertices[1:]-vertices[0]\n        error=float(np.max(np.linalg.norm(measured-expected[:,None,:],axis=-1)))\n        if error>5e-5:raise ValueError(f\'Independent known-translation test failed: {error} m\')\n        export_report=json.loads((temporary/\'diagnostic_only/episode_000000.report.json\').read_text())\n        report={\'diagnostic_only\':True,\'synthetic_parameters\':True,\'challenge_episode_reconstructed\':False,\n            \'frames\':frames,\'max_known_translation_error_m\':error,\n            \'native_round_trip_error_m\':export_report[\'max_native_round_trip_error_m\'],\n            \'mhr_model_sha256\':export_report[\'mhr_model_sha256\'],\'fixture_arrays_retained\':False}\n    a.report.parent.mkdir(parents=True,exist_ok=True)\n    a.report.write_text(json.dumps(report,indent=2)+\'\\n\');print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n')
mesh=next(Path('/kaggle/input').rglob('normalized_shape.glb'))
subprocess.run([py,str(code/'check_native_export.py'),'--assets',str(weights/'sam3d_body'),
 '--actual-mesh',str(mesh),'--report','/kaggle/working/native_export_diagnostic.json'],env=env,check=True)
